# Notebook 7 — Grounded RAG Assistant

This notebook builds the retrieval and answer-generation layer for the frozen electricity-price forecasting system.

It deliberately separates two kinds of knowledge:

1. **Static knowledge** — architecture, features, metric definitions, historical evaluation and limitations. It is embedded once and saved as a reusable index.
2. **Dynamic knowledge** — the previous scheduled forecast, corresponding actual prices and its completed error metrics. It is read directly from structured backend data and is never embedded as permanent knowledge.

Core guarantees:

- no model training or electricity-price prediction occurs here;
- no LLM arithmetic is used for MAE, RMSE, sMAPE or guarded MAPE;
- historical example data cannot silently masquerade as live data;
- answers must cite one or more supplied evidence identifiers;
- unsupported or insufficiently grounded questions receive a refusal;
- the Gemini API key is read from Kaggle Secrets or an environment variable and is never printed.


## 1. Install the current Google GenAI SDK

Add a Kaggle secret named `GEMINI_API_KEY` before running the API cells. The notebook uses the current `google-genai` package, the Gemini Interactions API for structured answers, and Gemini Embedding 2 for retrieval vectors.


In [1]:
%pip install -q --upgrade-strategy only-if-needed "google-genai==2.25.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.58.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you

In [2]:
import os
import re
import json
import time
import hashlib

from pathlib import Path
from typing import Literal, Optional

import numpy as np
import pandas as pd

from pydantic import BaseModel, Field
from google import genai
from google.genai import types


## 2. Configuration

Only the two static files below are indexed. The operational example is loaded solely for schema tests and is excluded from live answers by default.


In [3]:
PROJECT_INPUT_ROOT = Path(
    "/kaggle/input/notebooks/notjustauser"
)

STATIC_DOCUMENT_PATH = (
    PROJECT_INPUT_ROOT
    / "06-evaluation"
    / "static_model_evaluation.md"
)

MODEL_CONFIG_PATH = (
    PROJECT_INPUT_ROOT
    / "05-cnn-lstm"
    / "cnn_lstm_config.json"
)

HISTORICAL_EXAMPLE_PATH = (
    PROJECT_INPUT_ROOT
    / "06-evaluation"
    / "operational_evaluation_example.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/rag_assets"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GENERATION_MODEL = "gemini-3.8-flash"
EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIMENSION = 768

TOP_K = 4
MIN_RETRIEVAL_SCORE = 0.30
MAX_CHUNK_CHARACTERS = 1600

STATIC_INDEX_PATH = (
    OUTPUT_DIR / "static_embeddings.npy"
)

STATIC_CHUNKS_PATH = (
    OUTPUT_DIR / "static_chunks.json"
)

INDEX_METADATA_PATH = (
    OUTPUT_DIR / "static_index_metadata.json"
)


## 3. Load and validate the curated sources

The notebook uses an explicit allowlist. It does not scan directories or ingest every Markdown, CSV, Parquet or JSON file it finds.


In [4]:
STATIC_SOURCE_PATHS = {
    "static_model_evaluation": STATIC_DOCUMENT_PATH,
    "cnn_lstm_config": MODEL_CONFIG_PATH,
}

for source_name, source_path in STATIC_SOURCE_PATHS.items():
    if not source_path.exists():
        raise FileNotFoundError(
            f"Missing static source '{source_name}': "
            f"{source_path}"
        )

static_markdown = (
    STATIC_DOCUMENT_PATH.read_text(
        encoding="utf-8"
    )
)

with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:
    model_config = json.load(file)

required_config = {
    "lookback": 168,
    "horizon": 24,
    "conv1_filters": 32,
    "conv2_filters": 32,
    "lstm_units": 64,
}

for key, expected_value in required_config.items():
    received_value = model_config.get(key)

    if received_value != expected_value:
        raise ValueError(
            f"Invalid model configuration for '{key}'. "
            f"Expected {expected_value}, received "
            f"{received_value}."
        )

if "dynamic_data: false" not in static_markdown:
    raise ValueError(
        "The static document is not explicitly marked "
        "as static data."
    )

print("Static model document characters:", len(static_markdown))
print("Model version:", model_config.get("model_version", "cnn_lstm_frozen"))


Static model document characters: 7780
Model version: cnn_lstm_v1_frozen


In [5]:
historical_example_payload = None

if HISTORICAL_EXAMPLE_PATH.exists():
    with open(
        HISTORICAL_EXAMPLE_PATH,
        "r",
        encoding="utf-8"
    ) as file:
        historical_example_payload = json.load(file)

    if historical_example_payload.get("data_class") != "historical_example":
        raise ValueError(
            "The Notebook 6 example must be marked "
            "data_class='historical_example'."
        )

    if historical_example_payload.get("is_live") is not False:
        raise ValueError(
            "The Notebook 6 example must have is_live=false."
        )

    print("Historical operational example loaded for tests only.")
else:
    print(
        "Historical example not attached. Static RAG can still be built."
    )


Historical operational example loaded for tests only.


## 4. Authenticate without exposing the key

In Kaggle, create a secret called `GEMINI_API_KEY`. Outside Kaggle, use the environment variable with the same name.


In [6]:
def load_gemini_api_key():
    api_key = None

    try:
        from kaggle_secrets import UserSecretsClient

        api_key = (
            UserSecretsClient()
            .get_secret("GEMINI_API_KEY")
        )
    except Exception:
        api_key = os.environ.get(
            "GEMINI_API_KEY"
        )

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY was not found. Add it to "
            "Kaggle Secrets or the environment."
        )

    return api_key


GEMINI_API_KEY = load_gemini_api_key()

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client initialized. API key was not displayed.")


Gemini client initialized. API key was not displayed.


## 5. Create stable, section-aware chunks

Markdown is split by headings instead of arbitrary character windows. Model configuration is converted into deterministic key-value evidence. This keeps retrieved passages clear and reduces ambiguity.


In [7]:
def slugify(value):
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "section"


def split_long_text(text, max_characters=MAX_CHUNK_CHARACTERS):
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", text)
        if paragraph.strip()
    ]

    pieces = []
    current = []
    current_length = 0

    for paragraph in paragraphs:
        paragraph_length = len(paragraph)

        if current and current_length + paragraph_length > max_characters:
            pieces.append("\n\n".join(current))
            current = []
            current_length = 0

        if paragraph_length > max_characters:
            words = paragraph.split()
            temporary = []
            temporary_length = 0

            for word in words:
                if temporary and temporary_length + len(word) + 1 > max_characters:
                    pieces.append(" ".join(temporary))
                    temporary = []
                    temporary_length = 0

                temporary.append(word)
                temporary_length += len(word) + 1

            if temporary:
                pieces.append(" ".join(temporary))
        else:
            current.append(paragraph)
            current_length += paragraph_length + 2

    if current:
        pieces.append("\n\n".join(current))

    return pieces


def chunk_markdown(markdown_text, source_id):
    # Remove YAML front matter from the retrievable text.
    markdown_text = re.sub(
        r"\A---\s*\n.*?\n---\s*\n",
        "",
        markdown_text,
        flags=re.DOTALL,
    )

    heading_pattern = re.compile(
        r"^(#{1,3})\s+(.+?)\s*$",
        flags=re.MULTILINE,
    )

    matches = list(
        heading_pattern.finditer(markdown_text)
    )

    chunks = []

    for position, match in enumerate(matches):
        heading = match.group(2).strip()
        body_start = match.end()
        body_end = (
            matches[position + 1].start()
            if position + 1 < len(matches)
            else len(markdown_text)
        )

        body = markdown_text[body_start:body_end].strip()

        if not body:
            continue

        pieces = split_long_text(body)

        for piece_number, piece in enumerate(pieces, start=1):
            chunk_id = (
                f"{source_id}:"
                f"{slugify(heading)}:"
                f"{piece_number}"
            )

            chunks.append(
                {
                    "chunk_id": chunk_id,
                    "source_id": source_id,
                    "title": heading,
                    "text": f"{heading}\n\n{piece}",
                }
            )

    return chunks


In [8]:
def flatten_json(value, prefix=""):
    rows = []

    if isinstance(value, dict):
        for key in sorted(value):
            child_prefix = f"{prefix}.{key}" if prefix else key
            rows.extend(
                flatten_json(
                    value[key],
                    child_prefix
                )
            )
    elif isinstance(value, list):
        rendered = ", ".join(
            str(item) for item in value
        )
        rows.append(f"{prefix}: {rendered}")
    else:
        rows.append(f"{prefix}: {value}")

    return rows


static_chunks = chunk_markdown(
    static_markdown,
    source_id="static_model_evaluation"
)

config_text = (
    "Frozen CNN-LSTM configuration\n\n"
    + "\n".join(
        flatten_json(model_config)
    )
)

for piece_number, piece in enumerate(
    split_long_text(config_text),
    start=1
):
    static_chunks.append(
        {
            "chunk_id": (
                "cnn_lstm_config:model_contract:"
                f"{piece_number}"
            ),
            "source_id": "cnn_lstm_config",
            "title": "Frozen model contract",
            "text": piece,
        }
    )

unique_chunks = []
seen_hashes = set()

for chunk in static_chunks:
    text_hash = hashlib.sha256(
        chunk["text"].encode("utf-8")
    ).hexdigest()

    if text_hash not in seen_hashes:
        chunk["text_sha256"] = text_hash
        unique_chunks.append(chunk)
        seen_hashes.add(text_hash)

static_chunks = unique_chunks

if not static_chunks:
    raise ValueError("No static chunks were created.")

print("Static chunks:", len(static_chunks))

for chunk in static_chunks[:5]:
    print(chunk["chunk_id"], "|", len(chunk["text"]), "characters")


Static chunks: 14
static_model_evaluation:document_purpose:1 | 246 characters
static_model_evaluation:production_configuration:1 | 363 characters
static_model_evaluation:cnn_lstm_architecture:1 | 242 characters
static_model_evaluation:historical_evaluation_period:1 | 239 characters
static_model_evaluation:full_historical_test_metrics:1 | 627 characters


## 6. Build the static embedding index once

The index is reused when the source hash, embedding model and output dimension match. Dynamic daily data is intentionally excluded.


In [9]:
def normalize_vector(vector):
    vector = np.asarray(
        vector,
        dtype=np.float32
    )

    magnitude = np.linalg.norm(vector)

    if not np.isfinite(magnitude) or magnitude == 0:
        raise ValueError("Invalid embedding vector.")

    return vector / magnitude


def source_fingerprint(chunks):
    digest = hashlib.sha256()

    for chunk in chunks:
        digest.update(
            chunk["chunk_id"].encode("utf-8")
        )
        digest.update(
            chunk["text_sha256"].encode("utf-8")
        )

    return digest.hexdigest()


def embed_with_retry(text, purpose, attempts=3):
    if purpose not in {
        "retrieval_document",
        "retrieval_query"
    }:
        raise ValueError("Unsupported embedding purpose.")

    instruction = (
        "Task: retrieve authoritative electricity-price "
        "forecasting documentation. "
        f"Input type: {purpose}.\n\n{text}"
    )

    last_error = None

    for attempt in range(attempts):
        try:
            result = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=instruction,
                config=types.EmbedContentConfig(
                    output_dimensionality=EMBEDDING_DIMENSION
                ),
            )

            values = result.embeddings[0].values
            return normalize_vector(values)
        except Exception as error:
            last_error = error

            if attempt + 1 < attempts:
                time.sleep(2 ** attempt)

    raise RuntimeError(
        "Embedding request failed after retries."
    ) from last_error


In [10]:
current_fingerprint = source_fingerprint(
    static_chunks
)

index_can_be_reused = False

if (
    STATIC_INDEX_PATH.exists()
    and STATIC_CHUNKS_PATH.exists()
    and INDEX_METADATA_PATH.exists()
):
    with open(
        INDEX_METADATA_PATH,
        "r",
        encoding="utf-8"
    ) as file:
        existing_metadata = json.load(file)

    index_can_be_reused = (
        existing_metadata.get("source_fingerprint")
        == current_fingerprint
        and existing_metadata.get("embedding_model")
        == EMBEDDING_MODEL
        and existing_metadata.get("embedding_dimension")
        == EMBEDDING_DIMENSION
    )

if index_can_be_reused:
    static_embeddings = np.load(
        STATIC_INDEX_PATH
    ).astype(np.float32)

    with open(
        STATIC_CHUNKS_PATH,
        "r",
        encoding="utf-8"
    ) as file:
        static_chunks = json.load(file)

    print("Reused the existing static index.")
else:
    embedding_rows = []

    for position, chunk in enumerate(static_chunks, start=1):
        embedding_rows.append(
            embed_with_retry(
                chunk["text"],
                purpose="retrieval_document"
            )
        )

        print(
            f"Embedded {position}/{len(static_chunks)}:",
            chunk["chunk_id"]
        )

    static_embeddings = np.vstack(
        embedding_rows
    ).astype(np.float32)

    np.save(
        STATIC_INDEX_PATH,
        static_embeddings
    )

    with open(
        STATIC_CHUNKS_PATH,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            static_chunks,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    index_metadata = {
        "schema_version": "1.0",
        "source_fingerprint": current_fingerprint,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "chunk_count": len(static_chunks),
        "contains_dynamic_data": False,
    }

    with open(
        INDEX_METADATA_PATH,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            index_metadata,
            file,
            indent=2,
            allow_nan=False,
        )

    print("Created a new static index.")

if static_embeddings.shape != (
    len(static_chunks),
    EMBEDDING_DIMENSION
):
    raise ValueError(
        "Static index shape does not match the chunk metadata."
    )


Embedded 1/14: static_model_evaluation:document_purpose:1
Embedded 2/14: static_model_evaluation:production_configuration:1
Embedded 3/14: static_model_evaluation:cnn_lstm_architecture:1
Embedded 4/14: static_model_evaluation:historical_evaluation_period:1
Embedded 5/14: static_model_evaluation:full_historical_test_metrics:1
Embedded 6/14: static_model_evaluation:historical_12_00_utc_metrics:1
Embedded 7/14: static_model_evaluation:historical_subset_evaluation:1
Embedded 8/14: static_model_evaluation:historical_subset_evaluation:2
Embedded 9/14: static_model_evaluation:metric_definitions:1
Embedded 10/14: static_model_evaluation:model_selection_interpretation:1
Embedded 11/14: static_model_evaluation:input_rules:1
Embedded 12/14: static_model_evaluation:methodological_limitations:1
Embedded 13/14: static_model_evaluation:static_and_dynamic_rag_separation:1
Embedded 14/14: cnn_lstm_config:model_contract:1
Created a new static index.


## 7. Hybrid retrieval

Semantic similarity is combined with a small lexical-overlap signal. This helps exact terms such as `RMSE`, `lookback`, `Conv1D` and `MAPE` without turning retrieval into a brittle keyword-only system.


In [11]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by",
    "for", "from", "how", "in", "is", "it", "of", "on",
    "or", "that", "the", "this", "to", "was", "what",
    "when", "where", "which", "with",
}


def lexical_tokens(text):
    return {
        token
        for token in re.findall(
            r"[a-zA-Z0-9_]+",
            text.lower()
        )
        if token not in STOPWORDS
    }


def lexical_overlap(question, document):
    query_tokens = lexical_tokens(question)

    if not query_tokens:
        return 0.0

    document_tokens = lexical_tokens(document)

    return len(
        query_tokens & document_tokens
    ) / len(query_tokens)


def retrieve_static(question, top_k=TOP_K):
    question = question.strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    query_embedding = embed_with_retry(
        question,
        purpose="retrieval_query"
    )

    semantic_scores = (
        static_embeddings
        @ query_embedding
    )

    results = []

    for position, chunk in enumerate(static_chunks):
        lexical_score = lexical_overlap(
            question,
            chunk["text"]
        )

        combined_score = (
            0.85 * float(semantic_scores[position])
            + 0.15 * lexical_score
        )

        results.append(
            {
                **chunk,
                "semantic_score": float(
                    semantic_scores[position]
                ),
                "lexical_score": float(
                    lexical_score
                ),
                "score": float(combined_score),
            }
        )

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]


In [12]:
retrieval_probe = retrieve_static(
    "What is the CNN-LSTM architecture and input lookback?"
)

for result in retrieval_probe:
    print(
        f'{result["score"]:.3f}',
        result["chunk_id"]
    )


0.876 static_model_evaluation:cnn_lstm_architecture:1
0.843 cnn_lstm_config:model_contract:1
0.828 static_model_evaluation:production_configuration:1
0.790 static_model_evaluation:model_selection_interpretation:1


## 8. Deterministic question routing

Live or yesterday-specific language triggers dynamic lookup. General architecture, feature, metric-definition and historical-evaluation questions use static retrieval. Off-topic questions are rejected before any Gemini request.


In [13]:
PROJECT_SCOPE_TERMS = {
    "electricity", "price", "forecast", "forecasting",
    "model", "cnn", "lstm", "cnn-lstm", "architecture",
    "feature", "features", "lookback", "horizon", "load",
    "mae", "rmse", "smape", "mape", "error", "errors",
    "actual", "prediction", "predicted", "metric", "metrics",
    "training", "validation", "test", "historical",
}

DYNAMIC_PHRASES = {
    "today", "tomorrow", "yesterday", "latest forecast",
    "current forecast", "previous forecast", "last forecast",
    "latest prediction", "current prediction", "yesterday's",
    "yesterdays", "previous day", "last day", "live metric",
    "latest metric", "current metric",
}

STATIC_TERMS = {
    "architecture", "feature", "features", "lookback",
    "horizon", "training", "validation", "historical",
    "limitation", "limitations", "definition", "defined",
    "conv1d", "dropout", "scaler", "scalers", "why",
    "explain", "meaning", "interpret", "compare",
}


def classify_question(question):
    normalized = question.lower().strip()

    if not normalized:
        return "unsupported"

    scope_match = any(
        term in normalized
        for term in PROJECT_SCOPE_TERMS
    )

    if not scope_match:
        return "unsupported"

    dynamic_match = any(
        phrase in normalized
        for phrase in DYNAMIC_PHRASES
    )

    static_match = any(
        term in normalized
        for term in STATIC_TERMS
    )

    if dynamic_match and static_match:
        return "mixed"

    if dynamic_match:
        return "dynamic"

    return "static"


In [14]:
assert classify_question(
    "What is the CNN-LSTM architecture?"
) == "static"

assert classify_question(
    "What was yesterday's RMSE?"
) == "dynamic"

assert classify_question(
    "Compare yesterday's RMSE with historical limitations"
) == "mixed"

assert classify_question(
    "Who will win the football match?"
) == "unsupported"

print("Question-routing tests passed.")


Question-routing tests passed.


## 9. Validate dynamic backend data

Normal operation accepts only `data_class="live_operational"`. A historical example is accepted only when a test explicitly sets `allow_historical_example=True`.


In [15]:
def validate_dynamic_payload(
    payload,
    allow_historical_example=False
):
    if payload is None:
        return {
            "available": False,
            "reason": "No dynamic payload was supplied."
        }

    if payload.get("schema_version") != "1.0":
        raise ValueError(
            "Unsupported dynamic payload schema."
        )

    data_class = payload.get("data_class")

    allowed_classes = {
        "live_operational"
    }

    if allow_historical_example:
        allowed_classes.add(
            "historical_example"
        )

    if data_class not in allowed_classes:
        return {
            "available": False,
            "reason": (
                "Historical example data is blocked "
                "from live answers."
            )
        }

    if data_class == "live_operational" and payload.get("is_live") is not True:
        raise ValueError(
            "Live operational data must have is_live=true."
        )

    if data_class == "historical_example" and payload.get("is_live") is not False:
        raise ValueError(
            "Historical example data must have is_live=false."
        )

    status = payload.get("status")

    if status not in {
        "complete",
        "pending"
    }:
        raise ValueError(
            "Dynamic status must be complete or pending."
        )

    rows = payload.get("rows", [])

    if len(rows) != 24:
        raise ValueError(
            "Dynamic payload must contain exactly 24 rows."
        )

    horizons = [
        row.get("horizon")
        for row in rows
    ]

    if horizons != list(range(1, 25)):
        raise ValueError(
            "Dynamic horizons must be exactly 1 through 24."
        )

    utc_timestamps = [
        row.get("delivery_time_utc")
        for row in rows
    ]

    if len(set(utc_timestamps)) != 24:
        raise ValueError(
            "Dynamic UTC delivery timestamps must be unique."
        )

    if status == "complete":
        metrics = payload.get("metrics")

        if not isinstance(metrics, dict):
            raise ValueError(
                "A complete evaluation requires metrics."
            )

        if metrics.get("evaluated_hours") != 24:
            raise ValueError(
                "Completed metrics must use all 24 hours."
            )

    if status == "pending" and payload.get("metrics") is not None:
        raise ValueError(
            "Pending evaluation must not contain final metrics."
        )

    return {
        "available": True,
        "reason": None,
        "payload": payload,
    }


## 10. Deterministic answers for dynamic error metrics

These answers bypass Gemini completely. The language model is therefore unable to change, recalculate or invent yesterday's numbers.


In [16]:
METRIC_DEFINITIONS = {
    "mae": (
        "mae_eur_mwh",
        "MAE",
        "EUR/MWh"
    ),
    "rmse": (
        "rmse_eur_mwh",
        "RMSE",
        "EUR/MWh"
    ),
    "smape": (
        "smape_pct",
        "sMAPE",
        "%"
    ),
    "mape": (
        "mape_guarded_pct",
        "guarded MAPE",
        "%"
    ),
}


def requested_metric_names(question):
    normalized = question.lower()
    requested = []

    if re.search(r"\bmae\b", normalized):
        requested.append("mae")

    if re.search(r"\brmse\b", normalized):
        requested.append("rmse")

    if re.search(r"\bsmape\b", normalized):
        requested.append("smape")

    # Do not treat the MAPE substring inside sMAPE as MAPE.
    without_smape = re.sub(
        r"\bsmape\b",
        "",
        normalized
    )

    if re.search(r"\bmape\b", without_smape):
        requested.append("mape")

    if any(
        phrase in normalized
        for phrase in {
            "all metrics",
            "error metrics",
            "error rates",
            "performance metrics",
        }
    ):
        requested = [
            "mae",
            "rmse",
            "smape",
            "mape",
        ]

    return list(dict.fromkeys(requested))


def deterministic_metric_answer(
    question,
    payload,
    allow_historical_example=False
):
    requested = requested_metric_names(
        question
    )

    if not requested:
        return None

    validation = validate_dynamic_payload(
        payload,
        allow_historical_example=allow_historical_example
    )

    if not validation["available"]:
        return {
            "answer": validation["reason"],
            "source_ids": [],
            "data_status": "unavailable",
            "route": "dynamic",
        }

    payload = validation["payload"]

    if payload["status"] == "pending":
        return {
            "answer": payload.get(
                "message",
                "The evaluation is still pending."
            ),
            "source_ids": [
                "dynamic:previous_evaluation"
            ],
            "data_status": "pending",
            "route": "dynamic",
        }

    metrics = payload["metrics"]
    rendered_metrics = []

    for metric_name in requested:
        field_name, label, unit = (
            METRIC_DEFINITIONS[metric_name]
        )

        value = metrics.get(field_name)

        if value is None:
            rendered_metrics.append(
                f"{label}: unavailable"
            )
        else:
            rendered_metrics.append(
                f"{label}: {float(value):.3f} {unit}"
            )

    historical_prefix = (
        "Historical schema test only: "
        if payload.get("data_class") == "historical_example"
        else ""
    )

    answer = (
        historical_prefix
        + f"For the completed forecast issued at "
        f"{payload['forecast_origin_utc']}, "
        + "; ".join(rendered_metrics)
        + "."
    )

    if "mape" in requested:
        answer += (
            " Guarded MAPE excluded "
            f"{metrics.get('mape_excluded_hours', 0)} "
            "hour(s) with |actual price| below the configured threshold."
        )

    return {
        "answer": answer,
        "source_ids": [
            (
                "dynamic:historical_example"
                if payload.get("data_class") == "historical_example"
                else "dynamic:previous_evaluation"
            )
        ],
        "data_status": "complete",
        "route": "dynamic",
    }


## 11. Grounded Gemini response schema and guardrails

Gemini receives only the retrieved evidence assembled by the application. It is instructed to refuse unsupported claims and return source identifiers from the supplied evidence list.


In [17]:
class GroundedAnswer(BaseModel):
    answer: str = Field(
        description=(
            "A concise answer supported only by the supplied evidence."
        )
    )

    source_ids: list[str] = Field(
        description=(
            "Evidence identifiers used to support the answer."
        )
    )

    data_status: Literal[
        "static",
        "complete",
        "pending",
        "unavailable",
        "insufficient_evidence",
    ]


GROUNDED_SYSTEM_RULES = '''
You are the grounded assistant for one German electricity-price
forecasting project.

Follow every rule:
1. Use only the EVIDENCE blocks supplied in this request.
2. Do not use general knowledge, web knowledge, assumptions or guesses.
3. Do not calculate or recalculate MAE, RMSE, sMAPE, MAPE, prices or
   other numeric values.
4. Preserve all numeric values and units exactly as supplied.
5. Historical results must be labelled historical.
6. Never describe historical_example data as live, latest, current or
   yesterday's production result.
7. If evidence is insufficient, say that the requested information is
   unavailable in the supplied project evidence.
8. Do not provide trading, bidding, investment or financial advice.
9. Return only source_ids that appear in AVAILABLE_SOURCE_IDS.
10. Keep the answer below 180 words unless the user explicitly asks
    for a detailed explanation.
'''.strip()


In [18]:
def dynamic_evidence_block(
    payload,
    allow_historical_example=False
):
    validation = validate_dynamic_payload(
        payload,
        allow_historical_example=allow_historical_example
    )

    if not validation["available"]:
        return None

    payload = validation["payload"]

    compact_dynamic = {
        "schema_version": payload.get("schema_version"),
        "data_class": payload.get("data_class"),
        "is_live": payload.get("is_live"),
        "status": payload.get("status"),
        "model_version": payload.get("model_version"),
        "forecast_origin_utc": payload.get("forecast_origin_utc"),
        "forecast_origin_local": payload.get("forecast_origin_local"),
        "delivery_start_utc": payload.get("delivery_start_utc"),
        "delivery_end_utc": payload.get("delivery_end_utc"),
        "expected_hours": payload.get("expected_hours"),
        "available_actual_hours": payload.get("available_actual_hours"),
        "metrics": payload.get("metrics"),
        "message": payload.get("message"),
        "metric_notes": payload.get("metric_notes"),
    }

    source_id = (
        "dynamic:historical_example"
        if payload.get("data_class") == "historical_example"
        else "dynamic:previous_evaluation"
    )

    return {
        "source_id": source_id,
        "title": "Previous scheduled forecast evaluation",
        "text": json.dumps(
            compact_dynamic,
            indent=2,
            allow_nan=False,
        ),
    }


def build_grounded_prompt(question, evidence_blocks):
    available_source_ids = [
        block["source_id"]
        for block in evidence_blocks
    ]

    rendered_evidence = []

    for block in evidence_blocks:
        rendered_evidence.append(
            "\n".join(
                [
                    f"EVIDENCE_ID: {block['source_id']}",
                    f"TITLE: {block['title']}",
                    "CONTENT:",
                    block["text"],
                ]
            )
        )

    return "\n\n".join(
        [
            GROUNDED_SYSTEM_RULES,
            "AVAILABLE_SOURCE_IDS:\n"
            + json.dumps(available_source_ids),
            "USER_QUESTION:\n" + question,
            "EVIDENCE:\n"
            + "\n\n---\n\n".join(rendered_evidence),
        ]
    )


In [19]:
NUMBER_PATTERN = re.compile(
    r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?"
)


def numeric_values(text):
    return [
        float(match.group(0))
        for match in NUMBER_PATTERN.finditer(text)
    ]


def has_unsupported_numbers(answer, question, evidence_blocks):
    answer_numbers = numeric_values(answer)

    evidence_text = question + " " + " ".join(
        block["text"]
        for block in evidence_blocks
    )

    evidence_numbers = numeric_values(
        evidence_text
    )

    for answer_number in answer_numbers:
        supported = any(
            np.isclose(
                answer_number,
                evidence_number,
                rtol=5e-3,
                atol=5e-3,
            )
            for evidence_number in evidence_numbers
        )

        if not supported:
            return True

    return False


def generate_grounded_answer(question, evidence_blocks, data_status):
    if not evidence_blocks:
        return {
            "answer": (
                "The requested information is unavailable "
                "in the supplied project evidence."
            ),
            "source_ids": [],
            "data_status": "insufficient_evidence",
        }

    prompt = build_grounded_prompt(
        question,
        evidence_blocks
    )

    interaction = client.interactions.create(
        model=GENERATION_MODEL,
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": GroundedAnswer.model_json_schema(),
        },
    )

    parsed = GroundedAnswer.model_validate_json(
        interaction.output_text
    )

    available_source_ids = {
        block["source_id"]
        for block in evidence_blocks
    }

    invalid_sources = (
        set(parsed.source_ids)
        - available_source_ids
    )

    if invalid_sources:
        raise ValueError(
            "Gemini returned source identifiers "
            f"that were not supplied: {invalid_sources}"
        )

    if not parsed.source_ids:
        return {
            "answer": (
                "The requested information is unavailable "
                "in the supplied project evidence."
            ),
            "source_ids": [],
            "data_status": "insufficient_evidence",
        }

    if has_unsupported_numbers(
        parsed.answer,
        question,
        evidence_blocks
    ):
        return {
            "answer": (
                "The generated answer contained a numeric claim "
                "that was not present in the supplied evidence, "
                "so it was blocked."
            ),
            "source_ids": [],
            "data_status": "insufficient_evidence",
        }

    return {
        "answer": parsed.answer.strip(),
        "source_ids": parsed.source_ids,
        "data_status": data_status,
    }


## 12. Unified RAG entry point

Metric-only questions about yesterday are answered deterministically. Static and mixed explanatory questions use retrieval plus Gemini. Historical example data remains blocked unless a test explicitly enables it.


In [20]:
def answer_question(
    question,
    dynamic_payload=None,
    allow_historical_example=False
):
    question = question.strip()
    route = classify_question(question)

    if route == "unsupported":
        return {
            "answer": (
                "I can only answer questions about this "
                "electricity-price forecasting model, its "
                "documented evaluation and available operational metrics."
            ),
            "source_ids": [],
            "data_status": "unavailable",
            "route": route,
        }

    if route in {"dynamic", "mixed"}:
        direct_answer = deterministic_metric_answer(
            question,
            dynamic_payload,
            allow_historical_example=allow_historical_example
        )

        if direct_answer is not None and route == "dynamic":
            return direct_answer

    evidence_blocks = []

    if route in {"static", "mixed"}:
        retrieval_results = retrieve_static(
            question
        )

        accepted_results = [
            result
            for result in retrieval_results
            if result["score"] >= MIN_RETRIEVAL_SCORE
        ]

        if not accepted_results and route == "static":
            return {
                "answer": (
                    "The requested information is unavailable "
                    "in the indexed static project documentation."
                ),
                "source_ids": [],
                "data_status": "insufficient_evidence",
                "route": route,
            }

        for result in accepted_results:
            evidence_blocks.append(
                {
                    "source_id": result["chunk_id"],
                    "title": result["title"],
                    "text": result["text"],
                }
            )

    dynamic_status = "static"

    if route in {"dynamic", "mixed"}:
        dynamic_block = dynamic_evidence_block(
            dynamic_payload,
            allow_historical_example=allow_historical_example
        )

        if dynamic_block is None:
            if route == "dynamic":
                return {
                    "answer": (
                        "Current operational evaluation data "
                        "is not available."
                    ),
                    "source_ids": [],
                    "data_status": "unavailable",
                    "route": route,
                }
        else:
            evidence_blocks.append(dynamic_block)

            dynamic_status = (
                dynamic_payload.get(
                    "status",
                    "unavailable"
                )
            )

    response = generate_grounded_answer(
        question,
        evidence_blocks,
        data_status=dynamic_status
    )

    response["route"] = route
    return response


## 13. Safety and regression tests

The first tests do not call Gemini. They verify routing, historical-data blocking, strict JSON compatibility and deterministic metric answers.


In [21]:
if historical_example_payload is not None:
    blocked = validate_dynamic_payload(
        historical_example_payload,
        allow_historical_example=False
    )

    assert blocked["available"] is False

    allowed_for_test = validate_dynamic_payload(
        historical_example_payload,
        allow_historical_example=True
    )

    assert allowed_for_test["available"] is True

    metric_test = deterministic_metric_answer(
        "What were yesterday's MAE and RMSE?",
        historical_example_payload,
        allow_historical_example=True
    )

    assert metric_test["data_status"] == "complete"
    assert "MAE" in metric_test["answer"]
    assert "RMSE" in metric_test["answer"]

    json.dumps(
        historical_example_payload,
        allow_nan=False
    )

    print("Dynamic-data guardrail tests passed.")
else:
    print("Skipped historical-example tests because the file was not attached.")


Dynamic-data guardrail tests passed.


In [22]:
STATIC_RETRIEVAL_TESTS = [
    (
        "What is the CNN-LSTM architecture?",
        {"static_model_evaluation", "cnn_lstm_config"}
    ),
    (
        "Which 15 features are used by the model?",
        {"static_model_evaluation", "cnn_lstm_config"}
    ),
    (
        "Why is guarded MAPE used?",
        {"static_model_evaluation"}
    ),
    (
        "What are the methodological limitations?",
        {"static_model_evaluation"}
    ),
]

retrieval_test_rows = []

for question, expected_sources in STATIC_RETRIEVAL_TESTS:
    results = retrieve_static(question)
    returned_sources = {
        result["source_id"]
        for result in results
        if result["score"] >= MIN_RETRIEVAL_SCORE
    }

    passed = bool(
        returned_sources & expected_sources
    )

    retrieval_test_rows.append(
        {
            "question": question,
            "passed": passed,
            "top_score": (
                results[0]["score"]
                if results
                else None
            ),
            "sources": sorted(returned_sources),
        }
    )

retrieval_test_results = pd.DataFrame(
    retrieval_test_rows
)

display(retrieval_test_results)

if not retrieval_test_results["passed"].all():
    raise AssertionError(
        "One or more static retrieval tests failed. "
        "Inspect chunking and retrieval thresholds before deployment."
    )


,question,passed,top_score,sources
0,What is the CNN-LSTM architecture?,True,0.927321,"[cnn_lstm_config, static_model_evaluation]"
1,Which 15 features are used by the model?,True,0.777210,"[cnn_lstm_config, static_model_evaluation]"
2,Why is guarded MAPE used?,True,0.812940,[static_model_evaluation]
3,What are the methodological limitations?,True,0.909394,[static_model_evaluation]


## 14. Optional end-to-end Gemini tests

These cells consume generation quota. The historical example is enabled only for testing and the returned response must continue to identify its supplied evidence source.


In [23]:
static_answer = answer_question(
    "Explain the model architecture and its 168-hour lookback."
)

display(static_answer)


{'answer': 'The model architecture is a CNN-LSTM configured as follows:\n- First Conv1D layer: 32 filters, kernel size 3, causal padding\n- Pooling: pool size 2\n- Second Conv1D layer: 32 filters, kernel size 3\n- LSTM layer: 64 units, dropout 0.2\n- Dense layer: 64 units, dropout 0.2\n- Output: 24 forecast outputs (horizon 24, target price_eur_mwh)\n- Loss: Huber; Optimizer: Adam (learning rate 0.001)\n\nLookback details:\n- The model has an input shape of (168, 15), taking 168 engineered hourly rows across 15 features.\n- Exactly 168 complete engineered hourly rows are required for production forecasting.\n- A minimum of 336 continuous raw history hours is required because feature calculations need a 168-hour warm-up period.',
 'source_ids': ['cnn_lstm_config:model_contract:1',
  'static_model_evaluation:cnn_lstm_architecture:1',
  'static_model_evaluation:input_rules:1'],
 'data_status': 'static',
 'route': 'static'}

In [24]:
if historical_example_payload is not None:
    historical_metric_demo = answer_question(
        "What were yesterday's MAE, RMSE, sMAPE and MAPE?",
        dynamic_payload=historical_example_payload,
        allow_historical_example=True
    )

    # This is a historical schema test, not a live answer.
    display(historical_metric_demo)


{'answer': 'Historical schema test only: For the completed forecast issued at 2024-08-31T12:00:00+00:00, MAE: 14.585 EUR/MWh; RMSE: 17.441 EUR/MWh; sMAPE: 55.980 %; guarded MAPE: 70.120 %. Guarded MAPE excluded 2 hour(s) with |actual price| below the configured threshold.',
 'source_ids': ['dynamic:historical_example'],
 'data_status': 'complete',
 'route': 'dynamic'}

## 15. Export deployable RAG assets

The backend needs the static chunks, embeddings, index metadata, RAG configuration and guardrail prompt. It must obtain dynamic operational data from storage at request time.


In [25]:
RAG_CONFIG_PATH = (
    OUTPUT_DIR / "rag_config.json"
)

SYSTEM_PROMPT_PATH = (
    OUTPUT_DIR / "rag_system_rules.txt"
)

RAG_MANIFEST_PATH = (
    OUTPUT_DIR / "rag_manifest.json"
)

rag_config = {
    "schema_version": "1.0",
    "generation_model": GENERATION_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "top_k": TOP_K,
    "minimum_retrieval_score": MIN_RETRIEVAL_SCORE,
    "static_sources": sorted(STATIC_SOURCE_PATHS),
    "dynamic_data_is_embedded": False,
    "historical_examples_allowed_in_live_answers": False,
    "metric_calculation_by_llm": False,
    "supported_dynamic_schema": "1.0",
}

with open(
    RAG_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        rag_config,
        file,
        indent=2,
        allow_nan=False,
    )

SYSTEM_PROMPT_PATH.write_text(
    GROUNDED_SYSTEM_RULES,
    encoding="utf-8"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


asset_paths = [
    STATIC_INDEX_PATH,
    STATIC_CHUNKS_PATH,
    INDEX_METADATA_PATH,
    RAG_CONFIG_PATH,
    SYSTEM_PROMPT_PATH,
]

rag_manifest = {
    "schema_version": "1.0",
    "model_version": model_config.get(
        "model_version",
        "cnn_lstm_frozen"
    ),
    "contains_live_data": False,
    "assets": {
        path.name: {
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }
        for path in asset_paths
    },
}

with open(
    RAG_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        rag_manifest,
        file,
        indent=2,
        allow_nan=False,
    )

print("RAG assets exported to:", OUTPUT_DIR)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")


RAG assets exported to: /kaggle/working/rag_assets
rag_config.json 440 bytes
rag_manifest.json 848 bytes
rag_system_rules.txt 895 bytes
static_chunks.json 11861 bytes
static_embeddings.npy 43136 bytes
static_index_metadata.json 249 bytes


## 16. Final deployment rules

The deployed backend should:

- load the saved static embeddings and chunks at startup;
- read the latest dynamic evaluation record from storage for each relevant question;
- pass `allow_historical_example=False` in production;
- answer direct metric questions without Gemini;
- call Gemini only for grounded explanations requiring retrieved static evidence;
- return `pending` when fewer than 24 actual prices are available;
- never rebuild the static embedding index during the daily forecasting job;
- never expose the Gemini API key to the frontend;
- keep the frontend connected only to a protected backend API route.

The operational example produced by Notebook 6 is a schema fixture. It is not a substitute for backend data.
